In [ ]:
%%capture
!pip install --upgrade scikit-learn
!pip install --upgrade gensim
!apt-get install git

In [ ]:
%%capture
!git clone https://github.com/YJiangcm/SST-2-sentiment-analysis.git

In [ ]:
from google.colab import files
import pandas as pd
import re
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import gensim
import gensim.downloader

import seaborn as sns
import matplotlib.pyplot as plt

def Preprocess(Text):
    msg = re.sub('[^a-zA-Z]', ' ', Text)
    msg = msg.lower()
    msg = msg.split()
    msg = ' '.join(msg)
    return msg

In [ ]:
FTX = gensim.downloader.load('fasttext-wiki-news-subwords-300')

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
trainset = pd.read_csv("SST-2-sentiment-analysis/data/train.tsv", sep='\t', header=None, names=["class", "review"])
testset = pd.read_csv("SST-2-sentiment-analysis/data/test.tsv", sep='\t', header=None, names=["class", "review"])
trainset["p-review"] = trainset["review"].apply(Preprocess)
testset["p-review"] = testset["review"].apply(Preprocess)

In [ ]:
dim = FTX.vector_size
nTrain = round(len(trainset) * 0.8)

TrainVecs = []
TestVecs = []
print(dim)
for msg in trainset["p-review"]:
  words = msg.split()
  nwords = len(words)
  if nwords < 5:
    nwords = 5
  x = np.zeros((nwords, dim))
  for i in range(len(words)):
    if FTX.has_index_for(words[i]):
      x[i] = FTX[words[i]]
  TrainVecs.append(x)

DevVecs = TrainVecs[nTrain:]
TrainVecs = TrainVecs[:nTrain]

for msg in testset["p-review"]:
  words = msg.split()
  nwords = len(words)
  if nwords < 5:
    nwords = 5
  x = np.zeros((nwords, dim))
  for i in range(len(words)):
    if FTX.has_index_for(words[i]):
      x[i] = FTX[words[i]]
  TestVecs.append(x)


CTrain = trainset["class"][:nTrain]
CDev = trainset["class"][nTrain:]
CTest = testset["class"]
print(len(CTrain), len(CDev), len(CTest))

300
5536 1384 1821


In [ ]:
def toBatch(vecs, cls, batchsize=8, dim=dim):
  vecbatches = []
  clsbatches = []
  nvecs = len(vecs)
  for idx in range(nvecs // batchsize):
    maxlen = 0
    for i in range(batchsize):
      slen = len(vecs[idx * batchsize + i])
      if maxlen < slen:
        maxlen = slen
    vecbatch = np.zeros((batchsize, maxlen, dim))
    clsbatch = np.zeros((batchsize))
    for i in range(batchsize):
      clsbatch[i] = cls[idx * batchsize + i]
      slen = len(vecs[idx * batchsize + i])
      if slen < maxlen:
        pad = np.zeros((maxlen - slen, dim))
        vecbatch[i] = np.vstack((vecs[idx * batchsize + i], pad))
      else:
        vecbatch[i] = vecs[idx * batchsize + i]
    vecbatches.append(torch.FloatTensor(vecbatch))
    clsbatches.append(torch.LongTensor(clsbatch))
  return vecbatches, clsbatches


In [ ]:
VecBatches, ClsBatches = toBatch(TrainVecs, CTrain)
TVecBatches, TClsBatches = toBatch(TestVecs, CTest)

In [ ]:
DEVICE = "cpu"

class CNNClassifier(nn.Module):
  def __init__(self, hdim):
    super(CNNClassifier, self).__init__()
    cdim = hdim // 3
    self.cnns = nn.ModuleList([nn.Conv1d(300, cdim, 2), nn.Conv1d(300, cdim, 3), nn.Conv1d(300, cdim, 4)])
    self.fc = nn.Linear(hdim, 2)
    self.dropout = nn.Dropout(0.5)


  def forward(self, features, labels=None):
    x = x.transpose(1,2)
    hcnns = []
    for cnn in self.cnns:
      hcnn = torch.relu(cnn(x))
      pool = torch.max_pool1d(hcnn, hcnn.size(2)).squeeze(2)
      hcnns.append(pool)
    h = torch.cat(hcnns, dim=1)
    h = self.dropout(h)
    self.fc(h)
    loss = None
    if labels == None:
      loss = self.loss_func(logits, labels)
    return {"loss": loss, "logits": logits}

def train(mdl, vec, cls, ep=20):
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(mdl.parameters(), lr=0.001)
  steps = 0
  tloss = 0
  for i in range(ep):
    print("Epoch", i)
    for X,Y in zip(vec, cls):
      optimizer.zero_grad()
      out = mdl(X.to(DEVICE))
      loss = criterion(out, Y.to(DEVICE))
      tloss += loss.item()
      loss.backward()
      optimizer.step()
      steps += 1
      if steps % 100 == 0:
        print(f"Step {steps}, loss: {tloss/100}")
        tloss = 0



In [ ]:
model = CNNClassifier(300)

train(model, VecBatches, ClsBatches, 10)

Epoch 0
Step 100, loss: 0.685050675868988
Step 200, loss: 0.6632297280430793
Step 300, loss: 0.5910862803459167
Step 400, loss: 0.5437319299578667
Step 500, loss: 0.5115905192494392
Step 600, loss: 0.5271576753258705
Epoch 1
Step 700, loss: 0.4813959404826164
Step 800, loss: 0.4784075565636158
Step 900, loss: 0.4160655628144741
Step 1000, loss: 0.4035479271411896
Step 1100, loss: 0.43908787339925764
Step 1200, loss: 0.4120880277454853
Step 1300, loss: 0.4367524801939726
Epoch 2
Step 1400, loss: 0.4219435216486454
Step 1500, loss: 0.41015661977231505
Step 1600, loss: 0.35031020879745484
Step 1700, loss: 0.3557418678700924
Step 1800, loss: 0.387370343580842
Step 1900, loss: 0.3333600635826588
Step 2000, loss: 0.3826514220610261
Epoch 3
Step 2100, loss: 0.3558143404126167
Step 2200, loss: 0.34867625288665294
Step 2300, loss: 0.33913502357900144
Step 2400, loss: 0.31502846293151376
Step 2500, loss: 0.34899694822728633
Step 2600, loss: 0.29598500702530145
Step 2700, loss: 0.3133814929053187

In [ ]:
ncorrect = 0
model.train(False)
for X, Y in zip(TestVecs, CTest):
  output = model(torch.FloatTensor([X]))
  Y_ = output.argmax(dim=-1).detach().numpy()[0]
  if Y_ == Y:
    ncorrect += 1

print("Accuracy", ncorrect/len(CTest))

Accuracy 0.8396485447556288
